In [1]:
from ultralytics import YOLO, settings

In [2]:
# model = YOLO("yolo11n.pt")
# model = YOLO("yolo11n-seg.pt")
model = YOLO("yolo26n.pt")

In [ ]:
# IMPORTANT: COCO to YOLO Segmentation Conversion Script ONLY for yolo11n-seg.pt model
import json
import os
from pathlib import Path
from tqdm import tqdm

def convert_coco_to_yolo_seg(coco_json_path, output_dir):
    """Convert COCO JSON to YOLO segmentation format (polygons)."""
    
    with open(coco_json_path, 'r') as f:
        coco = json.load(f)
    
    os.makedirs(output_dir, exist_ok=True)
    
    images = {img['id']: img for img in coco['images']}
    categories = sorted(coco['categories'], key=lambda x: x['id'])
    cat_id_to_idx = {cat['id']: idx for idx, cat in enumerate(categories)}
    
    # Group annotations by image
    img_annotations = {}
    for ann in coco['annotations']:
        img_id = ann['image_id']
        if img_id not in img_annotations:
            img_annotations[img_id] = []
        img_annotations[img_id].append(ann)
    
    for img_id, img_info in tqdm(images.items(), desc="Converting"):
        img_w = img_info['width']
        img_h = img_info['height']
        filename = Path(img_info['file_name']).stem + '.txt'
        
        lines = []
        if img_id in img_annotations:
            for ann in img_annotations[img_id]:
                if ann.get('iscrowd', 0):
                    continue
                
                # Get segmentation polygons
                segmentation = ann.get('segmentation', [])
                
                # Skip if no segmentation or if it's RLE format
                if not segmentation or isinstance(segmentation, dict):
                    continue
                
                class_idx = cat_id_to_idx[ann['category_id']]
                
                # Process each polygon
                for polygon in segmentation:
                    if len(polygon) < 6:  # Need at least 3 points
                        continue
                    
                    # Normalize polygon coordinates
                    normalized = []
                    for i in range(0, len(polygon), 2):
                        x = polygon[i] / img_w
                        y = polygon[i + 1] / img_h
                        # Clamp to [0, 1]
                        x = max(0, min(1, x))
                        y = max(0, min(1, y))
                        normalized.extend([f"{x:.6f}", f"{y:.6f}"])
                    
                    line = f"{class_idx} " + " ".join(normalized)
                    lines.append(line)
        
        with open(os.path.join(output_dir, filename), 'w') as f:
            f.write('\n'.join(lines))
    
    print(f"✓ Converted {len(images)} images to {output_dir}")


# Run conversion
base = "/home/talt_wireten_c/road-segmentation/datasets/coco"

# Create new labels directory for segmentation
convert_coco_to_yolo_seg(
    f"{base}/annotations/instances_train2017.json",
    f"{base}/labels_seg/train2017"
)

convert_coco_to_yolo_seg(
    f"{base}/annotations/instances_val2017.json",
    f"{base}/labels_seg/val2017"
)

print("Done!")

Converting: 100%|██████████| 118287/118287 [00:27<00:00, 4252.12it/s]


✓ Converted 118287 images to /home/talt_wireten_c/road-segmentation/datasets/coco/labels_seg/train2017


Converting: 100%|██████████| 5000/5000 [00:01<00:00, 4193.96it/s]

✓ Converted 5000 images to /home/talt_wireten_c/road-segmentation/datasets/coco/labels_seg/val2017
Done!


In [ ]:
model.train(
    data="/home/talt_wireten_c/road-segmentation/config/my_coco.yaml",
    epochs=50,
    patience=10,
    imgsz=640,
    batch=32,
    optimizer="AdamW",
    device=3,
    workers=16,
    name="experiment_yolo26n"
)

New https://pypi.org/project/ultralytics/8.4.10 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.7 🚀 Python-3.10.19 torch-2.9.1+cu128 CUDA:3 (NVIDIA A100-SXM4-80GB, 81038MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/talt_wireten_c/road-segmentation/config/my_coco.yaml, degrees=0.0, deterministic=True, device=3, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=